# 04 — Baseline forecasting

One-month-ahead forecast of `total_waiting` for Trauma & Orthopaedics (C_110) at stable public providers.

**Sealed test year: FY 2025-26 (Apr 2025 – Mar 2026), declared 2026-07-29.** No cell below reads it; the final evaluation happens once, at the end of the project. Validation year: FY 2024-25. Two training regimes — full history vs post-Covid only — scored on the same validation rows. Everything here is deterministic (least squares, no sampling), so there is no seed to declare.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # arrays and least squares
from pathlib import Path

PROC = Path("../data/processed")
panel = pd.read_csv(PROC / "rtt_panel_with_provider_flags.csv",
                    parse_dates=["month", "first_month", "last_month"])

TEST_START = pd.Timestamp("2025-04-01")      # sealed test year starts here - never read below
VAL_START = pd.Timestamp("2024-04-01")       # validation year: FY 2024-25
COVID = pd.Timestamp("2020-03-01")           # first lockdown month, start of the second regime

dev = panel[panel["month"] < TEST_START]     # development window, Apr 2018 - Mar 2025
print("dev rows:", len(dev), "| sealed rows kept out:", int((panel["month"] >= TEST_START).sum()))

dev rows: 300193 | sealed rows kept out: 43248


## Working series and target

One row per provider-month; the value to predict is *next* month's waiting list. The last dev month per provider has no next month inside the window, so its target is empty and the row drops — that is also what keeps the sealed year out of the targets.

In [2]:
# Goal: C_110 (biggest specialty, 11.6% of the list) at stable public providers.
df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]].copy()
df = df.sort_values(["provider_code", "month"])

df["target"] = df.groupby("provider_code")["total_waiting"].shift(-1)   # next month's list
df["month_idx"] = (df["month"].dt.year - 2018) * 12 + df["month"].dt.month - 4   # months since Apr 2018
df = df.dropna(subset=["target"])            # each provider's final dev month: no next month exists

print(df["provider_code"].nunique(), "providers,", len(df), "provider-months")

130 providers, 10431 provider-months


In [3]:
# Goal: two training regimes, one validation year; both regimes end where validation
# begins, so no row is both trained on and scored.
train_full = df[df["month"] < VAL_START]                              # Apr 2018 - Mar 2024
train_post = df[(df["month"] >= COVID) & (df["month"] < VAL_START)]   # Mar 2020 - Mar 2024
val = df[df["month"] >= VAL_START]                                    # FY 2024-25 rows

assert val["month"].max() < TEST_START       # a sealed-year row leaking in would trip this
print(len(train_full), "train-full,", len(train_post), "train-post,", len(val), "validation rows")

9024 train-full, 6185 train-post, 1407 validation rows


## Naive baseline

The bar every model must beat: predict next month = this month. (Not "last value seen in training" — that goes stale across a 12-month validation year.)

In [4]:
truth = val["target"].to_numpy()
naive = val["total_waiting"].to_numpy()      # this month's list as the forecast for next month

print(f"naive: MAE {np.abs(truth - naive).mean():,.0f}   RMSE {np.sqrt(((truth - naive) ** 2).mean()):,.0f}   (n={len(val)})")

naive: MAE 153   RMSE 330   (n=1407)


## Pooled linear model

One least-squares fit shared by all providers: does knowing the long-wait mix and the date beat carrying forward the current number?

In [5]:
FEATS = ["total_waiting", "pct_gt18", "pct_gt52", "month_idx"]

def fit_lstsq(train):
    X = np.column_stack([np.ones(len(train)), train[FEATS].to_numpy()])   # leading 1s = intercept
    beta, *_ = np.linalg.lstsq(X, train["target"].to_numpy(), rcond=None)
    return beta

def predict(beta, rows):
    return np.column_stack([np.ones(len(rows)), rows[FEATS].to_numpy()]) @ beta

for name, tr in [("full-history", train_full), ("post-covid ", train_post)]:
    pred = predict(fit_lstsq(tr), val)
    print(f"linear {name}: MAE {np.abs(truth - pred).mean():,.0f}   RMSE {np.sqrt(((truth - pred) ** 2).mean()):,.0f}")

linear full-history: MAE 156   RMSE 330
linear post-covid : MAE 154   RMSE 330


In [6]:
# Goal: store the validation forecasts so the numbers above stay re-readable without a rerun.
out = val[["provider_code", "month", "target"]].copy()
out["pred_naive"] = naive
out["pred_lin_full"] = predict(fit_lstsq(train_full), val)
out["pred_lin_post"] = predict(fit_lstsq(train_post), val)
out.to_csv(PROC / "val_forecasts_c110_public_clients.csv", index=False)
print(out.shape, "->", PROC / "val_forecasts_c110_public_clients.csv")

(1407, 6) -> ..\data\processed\val_forecasts_c110_public_clients.csv
